# bathy2strat


In [1]:
# Start up: verify interpreter, install missing dependencies in this kernel, and ensure local imports work.
from pathlib import Path
import importlib
import subprocess
import sys

cwd = Path.cwd()
code_dir = cwd if (cwd / "bathy_formatter.py").exists() else (cwd / "code")
if code_dir.exists() and str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "netCDF4": "netCDF4",
    "pyproj": "pyproj",
    "geopandas": "geopandas",
}

missing_packages = []
for module_name, package_name in required_modules.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_name)

if missing_packages:
    missing_packages = sorted(set(missing_packages))
    print(f"Installing missing packages into this kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

# Re-check imports after install so failures are explicit.
for module_name in required_modules:
    importlib.import_module(module_name)

exe_norm = sys.executable.replace("\\", "/").lower()
if "/.venv/" not in exe_norm:
    print("WARNING: Notebook is not using the project .venv kernel.")
    print("Select kernel: Python (.venv bathy2strat)")

print(f"Python executable: {sys.executable}")
print("Startup dependency check complete.")

Select kernel: Python (.venv bathy2strat)
Python executable: c:\surf\500_Analysis\522_CarolinaInlets\scripts\python\.conda\python.exe
Startup dependency check complete.


## Bathymetry Formatter
Read in netcdf files and compile into bathymetry cube

In [2]:
from bathymetry_analysis import process_bathy_formatter, plot_bathy_formatter_outputs

# Define paths
nc_dir = r"C:\surf\300_Data\320_BogueInlet\netcdf\UTM_renamed"
out_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\data"
plot_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\plots\python2"

# Step 1: Process bathymetry (read, clean, interpolate, save outputs)
# Moderate shrink test: tighter than default, but avoids collapsing to blank outputs.
# output_format options: "netcdf" (CF-compliant, default), "mat" (MATLAB struct), "both"
bathy_result = process_bathy_formatter(
    nc_dir=nc_dir,
    out_dir=out_dir,
    dx=20.0,
    drop_year=2008,
    interp_method="linear",
    boundary_tightness=2.0,
    overlap_erosion_cells=1,
    output_format="netcdf",  # Change to "mat" or "both" as needed
)

bathy = bathy_result.bathy
print(f"Processing complete. Regridded bathymetry shape: {bathy.z.shape}")

Processing complete. Regridded bathymetry shape: (93, 92, 24)


# Plot output
Plot 

In [7]:
# Step 2: Generate QC plots from either in-memory result or saved file path
# Choose one source below:
plot_source = bathy_result
# plot_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.nc"
# plot_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.mat"

plot_bathy_formatter_outputs(
    process_result=plot_source,
    plot_dir=plot_dir,
    bathy_cmap_name="kg2",
    extents_cmap_name="viridis",
    extent_boundary_method="mask",
    boundary_tightness=2.0,
    overlap_erosion_cells=1,
    mask_nan_plots=True,
)

bathy = bathy_result.bathy
print(f"Plotting complete. Survey dates (MATLAB datenum): {bathy.t.flatten()}")
print(f"Grid spacing: x={bathy.x[0, 1] - bathy.x[0, 0]:.1f}m, y={bathy.y[1, 0] - bathy.y[0, 0]:.1f}m")

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'BathyProcessResult'

In [ ]:
# Utility: pick one source for all downstream analysis
# 1) in-memory process result
analysis_source = bathy_result
# 2) saved netCDF
# analysis_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.nc"
# 3) saved MAT
# analysis_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.mat"

import importlib
import bathy_formatter as bf

# Reload in case this module was imported before recent edits.
bf = importlib.reload(bf)

bathy_analysis = bf.load_for_analysis(analysis_source)
print(f"Analysis source type: {type(analysis_source).__name__}")
print(f"Analysis bathy shape (y, x, t): {bathy_analysis.z.shape}")

Analysis source type: BathyProcessResult
Analysis bathy shape (y, x, t): (93, 92, 24)


In [ ]:
from bathymetry_analysis import plot_bathy_formatter_outputs

plot_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\plots\python"
out_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\data"

nc_path = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.nc"
mat_path = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.mat"

# Test plotting directly from .nc path
plot_bathy_formatter_outputs(
    process_result=nc_path,
    plot_dir=plot_dir,
    bathy_cmap_name="kg2",
    extent_boundary_method="convex",
    mask_nan_plots=True,
)

# Test plotting directly from .mat path
plot_bathy_formatter_outputs(
    process_result=mat_path,
    plot_dir=plot_dir,
    bathy_cmap_name="kg2",
    extent_boundary_method="convex",
    mask_nan_plots=True,
)

print("Path-based plotting test complete for both .nc and .mat")

Loaded from file path: skipping raw extents/raw survey plots (source surveys unavailable).
Loaded from file path: skipping raw extents/raw survey plots (source surveys unavailable).
Path-based plotting test complete for both .nc and .mat


## Deposit Age Calculator
Calculate the age of deposits.


In [ ]:
from pathlib import Path
from bathymetry_analysis import StratigraphyConfig, run_stratigraphy_workflow

# Inputs (adjust as needed)
bathy_mat_path = mat_path  # from earlier cell; e.g. ..._bathyReGrid_..._py.mat
output_root = r"C:\surf\500_Analysis\522_CarolinaInlets"
shp_dir = r"C:\surfdrive\500_Analysis\522_CarolinaInlets\data\SHP"

# Configure the first-pass stratigraphy run
cfg = StratigraphyConfig(
    initial_index=0,
    dx=20.0,
    target_crs="EPSG:32618",
)

# Recommended first test: shapefile transects
result = run_stratigraphy_workflow(
    bathy_mat_path=bathy_mat_path,
    output_root=output_root,
    transect_mode="shapefile",
    shp_dir=shp_dir,
    source_crs=None,  # leave None to use shapefile CRS; prompted if missing
    config=cfg,
)

# Alternative modes (uncomment one if you want):
# result = run_stratigraphy_workflow(
#     bathy_mat_path=bathy_mat_path,
#     output_root=output_root,
#     transect_mode="gui",
#     config=cfg,
# )
# manual_transects = [np.array([[307900, 3835200], [308600, 3835900]], dtype=float)]
# result = run_stratigraphy_workflow(
#     bathy_mat_path=bathy_mat_path,
#     output_root=output_root,
#     transect_mode="manual",
#     manual_transects=manual_transects,
#     config=cfg,
# )

strat_plot_dir = Path(output_root) / "plots" / "python" / "stratigraphy"
strat_metrics_dir = Path(output_root) / "metrics" / "stratigraphy"

print("Stratigraphy run complete.")
print(f"deposit_per_year shape: {result.deposit_per_year.shape}")
print(f"theseus_ratio shape: {result.theseus_ratio.shape}")
print(f"plots: {strat_plot_dir}")
print(f"metrics: {strat_metrics_dir}")

c:\bathy2strat\code\bathymetry_analysis\stratigraphy.py:507: RuntimeWarning: All-NaN slice encountered
  y_min = min(np.nanmin(z_min) - 1.0, np.nanmin(baseline) - 1.0)
c:\bathy2strat\code\bathymetry_analysis\stratigraphy.py:508: RuntimeWarning: All-NaN slice encountered
  y_max = np.nanmax([np.nanmax(z_now) + 1.0, 3.0])
c:\bathy2strat\code\bathymetry_analysis\stratigraphy.py:512: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


Stratigraphy run complete.
deposit_per_year shape: (24, 24)
theseus_ratio shape: (24, 24)
plots: C:\surf\500_Analysis\522_CarolinaInlets\plots\python\stratigraphy
metrics: C:\surf\500_Analysis\522_CarolinaInlets\metrics\stratigraphy


## To do:
1. Convert Deposit Age Calculator (PLOTS ARE STILL EMPTY! COME BACK TO THIS!)
    
2. Convert plotting functions
3. Create dummy test with migrating Gaussian hump
4. Create theseus/preservation potential stuff
5. Make sure everything can run independently from command line etc
6. Compare with matlab version
7. Compare with hydrodynamics (cumulative wave power)
8. Make sure it can all run by itself (package/environment-wise)
9. Add pytest and linting CI/CD pipeline